The [representation learning](/aiml-common/lectures/vlm/representation-learning/index) section trained an encoder with a contrastive loss. Two augmented views of the same image formed a positive pair, the other images of the batch were negatives, and the augmentation decided which properties of the image the embedding kept. CLIP (Contrastive Language-Image Pretraining) keeps the loss and changes the views [@Radford2021-hl]. One view is an image, and the other is a piece of text written about it on the web, usually a caption or alt text. The embedding then keeps what the text talks about: the objects, their attributes and the scene, in the vocabulary of natural language.

This section shows the CLIP loss in code and measures it on a pretrained model, then explains the parts that decide how well it trains: the learnable temperature and the number of negatives in a batch. It then probes the known weaknesses of the embedding and ends with SigLIP, the variant whose image encoder is used by many current vision language models, including SmolVLM.

In [ ]:
import contextlib
import io
import os

import matplotlib.pyplot as plt
import transformers
from torchvision import datasets as _tv_datasets

plt.rcParams["figure.dpi"] = 110
os.environ["HF_HUB_DISABLE_PROGRESS_BARS"] = "1"
transformers.utils.logging.disable_progress_bar()
transformers.utils.logging.set_verbosity_error()
with contextlib.redirect_stderr(io.StringIO()), contextlib.redirect_stdout(io.StringIO()):
    _tv_datasets.CIFAR10("data", train=True, download=True)

In [ ]:
import numpy as np
import torch
import torch.nn.functional as F
from transformers import AutoModel, AutoProcessor

device = "cuda" if torch.cuda.is_available() else "cpu"
torch.manual_seed(0)
print(f"[INFO] device: {device}")

## Two encoders and one embedding space

CLIP has an image encoder $f_\ell$ and a text encoder $f_t$. The image encoder is a ResNet or a [vision transformer](/aiml-common/lectures/vision-transformers/vit/index), and the text encoder is a transformer that reads the caption. Each output passes through a linear projection to a shared dimension $d_z$ (512 for the model used here) and is normalized to unit length. Image $i$ becomes $\mathbf{z}_\ell^{(i)}$, caption $j$ becomes $\mathbf{z}_t^{(j)}$, and the cosine similarity of the two is their dot product.

<img src="images/clip_training_fig.png" alt="A batch of six image and caption pairs. Each image embedding is compared with each caption embedding in a 6 by 6 grid of dot products. The six diagonal entries are the true pairs (green), and the thirty off-diagonal entries are negatives (red)." style={{ width: "85%" }} />

*One training batch of six pairs. The diagonal holds the true pairs, and every other entry of the grid is a negative. Figure from Torralba, Isola and Freeman [@UnknownUnknown-zi], after Radford et al.*

For a batch of $N$ pairs, CLIP computes all $N^2$ similarities and multiplies them by a scale $s$:

$$
L_{ij} = s \, \mathbf{z}_\ell^{(i)} \cdot \mathbf{z}_t^{(j)}.
$$

Row $i$ of the matrix $L$ scores image $i$ against every caption, and the correct caption is in column $i$. That is a classification problem with $N$ classes, and the loss is the cross-entropy with target $i$. Column $j$ scores caption $j$ against every image, which is a second classification problem with target $j$. The CLIP loss is the average of the two:

$$
\mathcal{L} = \frac{1}{2N} \sum_{i=1}^{N} \left[ -\log \frac{e^{L_{ii}}}{\sum_{j} e^{L_{ij}}} \; - \; \log \frac{e^{L_{ii}}}{\sum_{j} e^{L_{ji}}} \right].
$$

Each half is the InfoNCE loss of the representation learning section, with the image as the anchor in the first and the caption as the anchor in the second. In code it is four lines.

In [ ]:
def clip_loss(z_image, z_text, scale):
    # z_image, z_text: (N, d) unit vectors, row i of each is a true pair
    logits = scale * z_image @ z_text.T                    # (N, N), L_ij
    targets = torch.arange(len(z_image), device=logits.device)
    return 0.5 * (F.cross_entropy(logits, targets) + F.cross_entropy(logits.T, targets))

## The loss on a pretrained model

The cell loads the CLIP model with a ViT-B/32 image encoder, released by OpenAI, and builds a batch of ten pairs from CIFAR-10: one test image of each class, with the caption "a photo of a {class}." The images are $32 \times 32$ and are enlarged to the $224 \times 224$ that the encoder expects, so they are harder to recognize than the photos CLIP was trained on.

In [ ]:
from torchvision import datasets

cifar = datasets.CIFAR10("data", train=False, download=True)
CLASSES = cifar.classes

# One test image per class, in class order
first = {}
for idx, label in enumerate(cifar.targets):
    first.setdefault(label, idx)
    if len(first) == 10:
        break
images = [cifar[first[c]][0] for c in range(10)]
captions = [f"a photo of a {c}." for c in CLASSES]

clip = AutoModel.from_pretrained("openai/clip-vit-base-patch32").to(device).eval()
clip_proc = AutoProcessor.from_pretrained("openai/clip-vit-base-patch32")


@torch.no_grad()
def clip_embed(images=None, texts=None):
    out = []
    if images is not None:
        x = clip_proc(images=images, return_tensors="pt").to(device)
        out.append(F.normalize(clip.get_image_features(**x).pooler_output, dim=-1))
    if texts is not None:
        x = clip_proc(text=texts, return_tensors="pt", padding=True).to(device)
        out.append(F.normalize(clip.get_text_features(**x).pooler_output, dim=-1))
    return out if len(out) > 1 else out[0]


z_img, z_txt = clip_embed(images, captions)
scale = clip.logit_scale.exp().item()
print(f"[INFO] learned scale s = exp(logit_scale) = {scale:.1f}")
print(f"[INFO] CLIP loss on this batch: {clip_loss(z_img, z_txt, scale).item():.3f}")
print(f"[INFO] loss of an untrained model (uniform softmax): log N = {np.log(10):.3f}")

In [ ]:
sim = (z_img @ z_txt.T).cpu().numpy()
fig = plt.figure(figsize=(8, 9.5))
gs = fig.add_gridspec(2, 1, height_ratios=[1, 6], hspace=0.25)
ax0 = fig.add_subplot(gs[0])
ax0.imshow(np.concatenate([np.array(im.resize((64, 64))) for im in images], axis=1))
ax0.set_xticks(32 + 64 * np.arange(10))
ax0.set_xticklabels(CLASSES, fontsize=8)
ax0.set_yticks([])
ax0.set_title("The ten images of the batch")
ax1 = fig.add_subplot(gs[1])
im = ax1.imshow(sim, cmap="viridis")
ax1.set_xticks(range(10))
ax1.set_xticklabels(CLASSES, rotation=45, ha="right", fontsize=8)
ax1.set_yticks(range(10))
ax1.set_yticklabels(CLASSES, fontsize=8)
ax1.set_xlabel("caption")
ax1.set_ylabel("image")
ax1.set_title("Cosine similarity of every image with every caption")
fig.colorbar(im, ax=ax1, shrink=0.8)
plt.show()

LOSS_PLACEHOLDER

## Temperature

The scale $s$ is the inverse of a temperature, $s = 1/\tau$. Cosine similarities lie between $-1$ and $1$, and in practice the similarities of one image with a set of captions differ by a few hundredths. Fed to a softmax as they are, they give nearly equal probabilities to every caption, and the cross-entropy cannot fall much below $\log N$ however good the embedding is. Multiplying by $s$ stretches these small differences.

CLIP learns $s$ along with the encoders. It is stored as $\log s$, initialized to $\log(1/0.07)$, and clipped so that $s$ never exceeds 100. The model used here ended training at that limit. The cell shows the probabilities that image 0 (an airplane) assigns to the ten captions at three values of $s$.

In [ ]:
row = z_img[0] @ z_txt.T
for s in (1.0, 10.0, scale):
    p = (s * row).softmax(dim=-1)
    print(f"s = {s:>5.1f}: p(correct caption) = {p[0].item():.3f}   largest wrong p = {p[1:].max().item():.3f}")

## Negatives and batch size

Every other pair in the batch is a negative, so a batch of $N$ pairs gives each image $N - 1$ negative captions. The loss of a model that has learned nothing is $\log N$, and the more negatives there are, the more the loss rewards fine distinctions: telling a photo of a dog from 9 random captions is easy, telling it from 32,767 captions, some of which describe other dogs, is not. CLIP was trained with batches of 32,768 pairs on 400 million image and text pairs collected from the web. Such batches do not fit on one GPU. Each GPU embeds its share of the batch, and all embeddings are gathered on every GPU to build the full $N \times N$ matrix.

A large batch also has a cost that the CIFAR-10 example makes visible. Some negatives are not really negative. In a batch with two photos of dogs, the caption of one also describes the other, and the loss pushes them apart anyway. With 32,768 web captions this matters less, because captions are rarely identical, but it is the reason why class-label captions such as the ones above make poor training data for CLIP.

## What the embedding is used for

After training, the two encoders are used separately or together. Zero-shot classification embeds one caption per class and assigns an image to the class whose caption is most similar; the [zero-shot section](/aiml-common/lectures/vlm/clip/zero-shot/index) builds this classifier and measures it. A *linear probe* trains a logistic regression on the frozen image embeddings with labeled examples, and measures how much information the embedding holds about a task. Retrieval ranks a collection of images by their similarity to a text query, or the reverse. Vision language models such as LLaVA keep only the image encoder and feed its patch features, not the single pooled embedding, to a language model.

## Failure modes

CLIP's embedding inherits the strengths and the gaps of its training signal. A caption names what is in a photo far more often than how many there are or where they are, and the loss rewards a caption embedding that identifies the right image among $N$ candidates, which a bag of the right words often does. Radford et al. report weak zero-shot accuracy on counting objects and on estimating distances, and on handwritten digits, which are rare on the web. Three more failures are easy to show with a few lines of code.

**Word order.** Yuksekgonul et al. found that CLIP and similar models often cannot tell "the horse is eating the grass" from "the grass is eating the horse", and behave like bags of words ([arXiv:2210.01936](https://arxiv.org/abs/2210.01936)). The cell compares pairs of captions with the same words in a different order.

In [ ]:
pairs = [
    ("a dog chasing a cat", "a cat chasing a dog"),
    ("a red cube on top of a blue sphere", "a blue cube on top of a red sphere"),
    ("a horse eating grass", "grass eating a horse"),
    ("a dog chasing a cat", "a photo of a dog"),
]
for a, b in pairs:
    za, zb = clip_embed(texts=[a, b])
    print(f"{(za @ zb).item():.3f}   '{a}'  vs  '{b}'")

ORDER_PLACEHOLDER

**The modality gap.** The two encoders are trained so that true pairs are more similar than false ones, not so that an image and its caption land on the same point. Liang et al. showed that the image embeddings and the text embeddings occupy two separate regions of the sphere, the *modality gap* ([arXiv:2203.02053](https://arxiv.org/abs/2203.02053)). The cell compares the average similarity within each modality and across them, for the ten images and captions above.

In [ ]:
off = ~torch.eye(10, dtype=torch.bool, device=device)
print(f"image-image, different images:      {(z_img @ z_img.T)[off].mean().item():.3f}")
print(f"text-text, different captions:      {(z_txt @ z_txt.T)[off].mean().item():.3f}")
print(f"image-text, true pairs:             {(z_img * z_txt).sum(1).mean().item():.3f}")
print(f"image-text, false pairs:            {(z_img @ z_txt.T)[off].mean().item():.3f}")
print(f"cosine of the two modality means:   {F.cosine_similarity(z_img.mean(0), z_txt.mean(0), dim=0).item():.3f}")

GAP_PLACEHOLDER

**Typographic attacks.** CLIP learned to read, because web images often contain text that matches their captions. Goh et al. showed that a handwritten label stuck on an apple can make CLIP classify it as the labeled object ([Multimodal Neurons in Artificial Neural Networks](https://distill.pub/2021/multimodal-neurons/)). The cell writes a class name in white letters on CIFAR-10 test images of other classes and checks how often the zero-shot prediction follows the word.

In [ ]:
from PIL import Image, ImageDraw, ImageFont

try:
    font = ImageFont.truetype("DejaVuSans-Bold.ttf", 34)
except OSError:
    font = ImageFont.load_default()


def write_word(img, word):
    big = img.resize((224, 224), Image.BICUBIC)
    draw = ImageDraw.Draw(big)
    draw.rectangle([0, 160, 224, 224], fill=(0, 0, 0))
    draw.text((112, 192), word, fill=(255, 255, 255), font=font, anchor="mm")
    return big


z_classes = clip_embed(texts=captions)
rng = np.random.default_rng(0)
idx = rng.choice(len(cifar), 200, replace=False)
clean = [cifar[i][0] for i in idx]
labels = np.array([cifar[i][1] for i in idx])
words = (labels + rng.integers(1, 10, len(labels))) % 10      # a different class for every image
attacked = [write_word(im, CLASSES[w]) for im, w in zip(clean, words)]

pred_clean = (clip_embed(images=clean) @ z_classes.T).argmax(1).cpu().numpy()
pred_attacked = (clip_embed(images=attacked) @ z_classes.T).argmax(1).cpu().numpy()
print(f"zero-shot accuracy on the clean images:   {(pred_clean == labels).mean():.2f}")
print(f"zero-shot accuracy with a wrong word:     {(pred_attacked == labels).mean():.2f}")
print(f"predictions that follow the written word: {(pred_attacked == words).mean():.2f}")

In [ ]:
fig, axes = plt.subplots(1, 6, figsize=(13, 2.6))
for ax, k in zip(axes, range(6)):
    ax.imshow(attacked[k])
    ax.set_title(f"true: {CLASSES[labels[k]]}\npred: {CLASSES[pred_attacked[k]]}", fontsize=8)
    ax.axis("off")
plt.tight_layout()
plt.show()

TYPO_PLACEHOLDER

**Bias.** The training pairs come from the web, and the embedding reflects how the web describes people. Radford et al. measured this themselves. When they added labels such as "criminal" and "animal" to a zero-shot classifier for photos of faces, a share of the photos were assigned to them, at rates that differed between racial groups and between age groups. A zero-shot classifier can be built for any list of words, including words that should not be applied to people, and nothing in the model prevents it.

## SigLIP: a sigmoid loss

The softmax in the CLIP loss couples all the pairs of a batch: the probability of the true caption depends on the scores of all $N$ captions, so the loss of one pair cannot be computed without the whole $N \times N$ matrix, and the result depends on the batch size. SigLIP replaces it with an independent binary decision for every entry of the matrix [@Zhai2023-lj]. Entry $(i, j)$ is a positive example if $i = j$ and a negative one otherwise, with label $y_{ij} = 1$ or $-1$:

$$
\mathcal{L} = -\frac{1}{N} \sum_{i=1}^{N} \sum_{j=1}^{N} \log \sigma\!\left( y_{ij} \left( s \, \mathbf{z}_\ell^{(i)} \cdot \mathbf{z}_t^{(j)} + b \right) \right).
$$

There is a learned scale $s$, as in CLIP, and a learned bias $b$. Because $N - 1$ of every $N$ entries are negatives, the loss would be dominated by them at the start of training, so $b$ is initialized to $-10$, which makes every pair start as a confident negative. Each entry is a separate term, so each GPU can compute the loss for its own block of the matrix and exchange embeddings with its neighbors, without a softmax over the whole batch. Zhai et al. found that the sigmoid loss beats the softmax loss at batch sizes below about 16,000, and that the benefit of larger batches levels off around 32,000.

The cell measures the loss with SigLIP 2, which keeps the sigmoid loss and adds further training objectives ([arXiv:2502.14786](https://arxiv.org/abs/2502.14786)). The same change shows up at inference. The scores of a SigLIP model are independent probabilities, one per image and caption, and they do not have to sum to one. An image can match several captions or none.

In [ ]:
def siglip_loss(z_image, z_text, scale, bias):
    logits = scale * z_image @ z_text.T + bias               # (N, N)
    labels = 2 * torch.eye(len(z_image), device=logits.device) - 1   # +1 on the diagonal, -1 elsewhere
    return -F.logsigmoid(labels * logits).sum() / len(z_image)


siglip = AutoModel.from_pretrained("google/siglip2-base-patch16-224").to(device).eval()
siglip_proc = AutoProcessor.from_pretrained("google/siglip2-base-patch16-224")

with torch.no_grad():
    x = siglip_proc(text=captions, images=images, padding="max_length", max_length=64, return_tensors="pt").to(device)
    out = siglip(**x)
    zs_img = F.normalize(out.image_embeds, dim=-1)
    zs_txt = F.normalize(out.text_embeds, dim=-1)
s, b = siglip.logit_scale.exp().item(), siglip.logit_bias.item()
print(f"[INFO] SigLIP scale s = {s:.1f}, bias b = {b:.1f}")
print(f"[INFO] SigLIP loss on the CIFAR-10 batch: {siglip_loss(zs_img, zs_txt, s, b).item():.3f}")

p_siglip = torch.sigmoid(s * zs_img @ zs_txt.T + b)
p_clip = (scale * z_img @ z_txt.T).softmax(dim=-1)
print(f"[INFO] images whose true caption has the highest SigLIP probability: "
      f"{(p_siglip.argmax(1) == torch.arange(10, device=device)).sum().item()} of 10")
for k in range(3):
    print(f"image {k} ({CLASSES[k]}): CLIP probabilities sum to {p_clip[k].sum().item():.2f}, "
          f"SigLIP probabilities sum to {p_siglip[k].sum().item():.2f}, "
          f"SigLIP p(correct) = {p_siglip[k, k].item():.2f}")

SIGLIP_PLACEHOLDER

## Exercises

1. Compute the CLIP loss of the CIFAR-10 batch with two images of the same class (for example two cats, with two identical captions). What happens to the loss, and why is this a false negative?
2. Train the contrastive encoder of the representation learning section with the temperature fixed at 1, 0.2 and 0.05. Plot the final loss and the embedding for each, and explain the role of the temperature.
3. Repeat the word-order test with SigLIP. Does the sigmoid loss help? Explain why the loss alone should not be expected to fix it.
4. Measure the typographic attack as a function of the size of the written word.

## Key references

Radford et al. introduced CLIP and evaluated it on more than 30 data sets, including its failures and biases [@Radford2021-hl]. Zhai et al. introduced the sigmoid loss of SigLIP [@Zhai2023-lj]. Chapter 51 of Torralba, Isola and Freeman covers vision and language [@UnknownUnknown-zi].